In [5]:
# Load packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
# Path

# skyline_path = 'pqp/PeptideLevel_OpenMS_8files.csv'
# ## Read data
# skyline_data = pd.read_csv(skyline_path)
# ## For skyline data, remove column Precursor and drop duplicates
# skyline_data = skyline_data.drop(columns=['Precursor'])



# SDRF 
sdrf_dir = 'data/MARTHA/sdrf'
# Print out all files in the directory
print(os.listdir(sdrf_dir))
# Read all file and combine them into one dataframe
sdrf_files = [f for f in os.listdir(sdrf_dir) if f.endswith('.tsv')]
# Reorder by name
sdrf_files.sort()

sdrf_data = pd.concat([pd.read_csv(os.path.join(sdrf_dir, f), sep='\t') for f in sdrf_files])
# Save the combined sdrf data to a file
# sdrf_data.to_csv('export/sdrf_MARTHA_combined.tsv', sep='\t', index=False)




['20251216_MarthaReselection_Plate_7.sdrf.tsv', '20251204_MarthaReselection_Plate_4.sdrf.tsv', '20251218_MarthaReselection_Plate_9.sdrf.tsv', '20251207_MarthaReselection_Plate_5.sdrf.tsv', '20251207_MarthaReselection_Plate_6.sdrf.tsv', '20251218_MarthaReselection_Plate_8.sdrf.tsv', '20251128_MarthaReselection_Plate_2.sdrf.tsv', '20251128_MarthaReselection_Plate_1.sdrf.tsv', '20251203_MarthaReselection_Plate_3.sdrf.tsv']


In [9]:
sdrf_data.head()

,source name,characteristics[Row],characteristics[Column],characteristics[Sample],characteristics[Source Vial],characteristics[Position],characteristics[Inj Vol],characteristics[Instrument Method],characteristics[Path],characteristics[File Name],...,comment[cleavage agent details],comment[ms2 mass analyzer],comment[instrument],comment[modification parameters],comment[dissociation method],comment[collision energy],comment[precursor mass tolerance],comment[fragment mass tolerance],factor value[Row],﻿source name
0,PRM_20251216_MarthaReselection_Plate_7_A1,A,1,MRTH,1,RA1,0.1,C:\Xcalibur\methods\method1,C:\data\yourdir,PRM_20251216_MarthaReselection_Plate_7_A1,...,AC=MS:1001251;NT=Trypsin,not available,NT=Stellar;AC=MS:1003409,NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed,NT=Higher-energy Collisional Dissociation;AC=M...,27 NCE,40 ppm,0.05 Da,A,NaN
1,PRM_20251216_MarthaReselection_Plate_7_A2,A,2,MRTH,2,RA2,0.1,C:\Xcalibur\methods\method1,C:\data\yourdir,PRM_20251216_MarthaReselection_Plate_7_A2,...,AC=MS:1001251;NT=Trypsin,not available,NT=Stellar;AC=MS:1003409,NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed,NT=Higher-energy Collisional Dissociation;AC=M...,27 NCE,40 ppm,0.05 Da,A,NaN
2,PRM_20251216_MarthaReselection_Plate_7_A3,A,3,MRTH,3,RA3,0.1,C:\Xcalibur\methods\method1,C:\data\yourdir,PRM_20251216_MarthaReselection_Plate_7_A3,...,AC=MS:1001251;NT=Trypsin,not available,NT=Stellar;AC=MS:1003409,NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed,NT=Higher-energy Collisional Dissociation;AC=M...,27 NCE,40 ppm,0.05 Da,A,NaN
3,PRM_20251216_MarthaReselection_Plate_7_A4,A,4,MRTH,4,RA4,0.1,C:\Xcalibur\methods\method1,C:\data\yourdir,PRM_20251216_MarthaReselection_Plate_7_A4,...,AC=MS:1001251;NT=Trypsin,not available,NT=Stellar;AC=MS:1003409,NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed,NT=Higher-energy Collisional Dissociation;AC=M...,27 NCE,40 ppm,0.05 Da,A,NaN
4,PRM_20251216_MarthaReselection_Plate_7_A5,A,5,MRTH,5,RA5,0.1,C:\Xcalibur\methods\method1,C:\data\yourdir,PRM_20251216_MarthaReselection_Plate_7_A5,...,AC=MS:1001251;NT=Trypsin,not available,NT=Stellar;AC=MS:1003409,NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed,NT=Higher-energy Collisional Dissociation;AC=M...,27 NCE,40 ppm,0.05 Da,A,NaN


In [ ]:
openswath_columns = ['filename', 'ProteinName', 'Sequence', 'FullPeptideName', 'Charge',  'peak_group_rank', 'Intensity']

# Select columns
openswath_data = openswath_data[openswath_columns]

# Update filename using regex with text between ./mzml/ and .mzML
openswath_data['filename'] = openswath_data['filename'].str.extract(r'./mzml/(.*).mzML')

openswath_data.head()

In [ ]:
# Create new column called 'Isotope Label Type' where it checks regex of FullPeptideName if it ends with R(Unimod:267) or R(Unimod:259)
# If yes, assign 'heavy' otherwise 'light'
openswath_data['Isotope Label Type'] = openswath_data['FullPeptideName'].apply(
    lambda x: 'heavy' if pd.notnull(x) and (x.endswith('(UniMod:267)') or x.endswith('(UniMod:259)')) else 'light'
)


# Rename columns 
openswath_data.rename(columns={'filename': 'Replicate', 
                             'ProteinName': 'Protein Name', 
                             'Sequence': 'Peptide', 
                             'Charge': 'Precursor Charge', 
                             'aggr_Peak_Area': 'OpenSwath Intensity', 
                             }, inplace=True)


In [ ]:
# Filter Peptide leveled and calculated ration

# Filtererd peak_group_rank is 1
openswath_data = openswath_data[openswath_data['peak_group_rank'] == 1]

# Fix: Use the correct column name for intensity ("Intensity" instead of "OpenSwath Intensity") and create separate columns for heavy and light
openswath_pivot = openswath_data.pivot_table(
    index=['Replicate', 'Protein Name', 'Peptide', 'Precursor Charge'],
    columns='Isotope Label Type',
    values='Intensity', 
    aggfunc='first'
).reset_index()

# Merge the additional columns back into the main data frame if necessary
openswath_pivot = openswath_pivot


# Create light ot heavy ratio
openswath_pivot['openswath_lh_ratio'] = openswath_pivot['light'] / openswath_pivot['heavy']


# Remove NaN values in lh_ratio
openswath_pivot = openswath_pivot[openswath_pivot['openswath_lh_ratio'].notna()]

openswath_pivot.head()

In [ ]:
ratio_openswath = openswath_pivot[['Replicate','Protein Name', 'Peptide', 'openswath_lh_ratio']]

ratio_openswath.head()

In [ ]:
# # Add column Isotope Label Type
# skyline_data['Isotope Label Type'] = skyline_data['Precursor'].apply(lambda x: 'heavy' if re.search(r'heavy', str(x), re.IGNORECASE) else 'light')

# # Remove Normalied Area that is Nan
# skyline_data = skyline_data[skyline_data['Normalized Area'].notna()]
# # Prepare pivot table, keeping all other column names as index except 'Isotope Label Type' and 'Intensity'
# index_cols = [col for col in skyline_data.columns if col not in ['Isotope Label Type', 'Intensity']]
# skyline_pivot = skyline_data.pivot_table(
#     index=['Replicate', 'Peptide'],
#     columns='Isotope Label Type',
#     values='Normalized Area',
#     aggfunc='first'
# ).reset_index()

# # Calculate lh_ratio
# skyline_pivot['lh_ratio'] = skyline_pivot['heavy'] / skyline_pivot['light']


# skyline_pivot.head(20)

In [ ]:
selected_columns = ['Replicate', 'Peptide', 'RatioLightToHeavy']
skyline_ratio = skyline_data[selected_columns]

# Remove row with NaN in RatioLightToHeavy
skyline_ratio = skyline_ratio[skyline_ratio['RatioLightToHeavy'].notna()]
# Order by Replicate and Peptide
skyline_ratio = skyline_ratio.sort_values(by=['Replicate', 'Peptide'])
skyline_ratio.head()


In [ ]:
skyline_ratio

In [ ]:
# scrape concentration data from the web

import requests
import pandas as pd

def fetch_qreps_table(link: str) -> pd.DataFrame:
    response = requests.get(link, timeout=30)
    response.raise_for_status()

    all_tables = pd.read_html(response.text)
    for table in all_tables:
        normalized_columns = [str(col).strip().lower() for col in table.columns]
        if 'qreps' in normalized_columns and 'amount per well [pmol]' in normalized_columns:
            return table

    raise ValueError("Could not find qRePS data table on the page.")

lot23002_link = 'https://proteomedge.com/lotdata/23002/'
qreps_df = fetch_qreps_table(lot23002_link)
qreps_df

In [ ]:
ratio_openswath.head()

In [ ]:
# Merge openswath and skyline ratio

# Merge openswath and skyline ratio
merged_data = pd.merge(ratio_openswath, skyline_ratio, on=['Replicate', 'Peptide'], how='outer')

# Merge with qreps data
# merged_data = pd.merge(merged_data, qreps_df, left_on=['Protein Name'], right_on=['Uniprot'], how='left')

merged_data.head()

In [ ]:
# overlapping peptides between methods


# Select columns 
matched_peptides = merged_data[['Replicate', 'Protein Name','Peptide', 'openswath_lh_ratio', 'RatioLightToHeavy']]
matched_peptides.head()

In [ ]:
# Select row where both ratio are not NaN
# Filter data to include rows where both ratios are not NaN
plot_merged_data = merged_data[merged_data['openswath_lh_ratio'].notna() & merged_data['RatioLightToHeavy'].notna()]

# Please remove ratio that is not in range from 10**-3 and 10**3
plot_merged_data = plot_merged_data[
    plot_merged_data['openswath_lh_ratio'].between(10**-3, 10**3) & 
    plot_merged_data['RatioLightToHeavy'].between(10**-3, 10**3)
]

# MASK replicate values with Sample_1 to Sample_8
unique_replicates_sorted = sorted(plot_merged_data['Replicate'].unique())
replicate_map = {rep: f"Sample_{i+1}" for i, rep in enumerate(unique_replicates_sorted)}
plot_merged_data['Replicate_masked'] = plot_merged_data['Replicate'].map(replicate_map)

# COLOR THE DOTS BY PROTEIN NAME
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# Ensure that the 'Protein Name' field exists in plot_merged_data; if not, fill with 'Unknown'
if 'Protein Name' not in plot_merged_data.columns:
    plot_merged_data['Protein Name'] = 'Unknown'

# Prepare colormap for protein names
protein_names = plot_merged_data['Protein Name'].astype(str)
unique_proteins = protein_names.unique()
cmap = cm.get_cmap('tab20', len(unique_proteins))
protein2color = {prot: cmap(i) for i, prot in enumerate(unique_proteins)}

# Use the masked replicate names
unique_replicates_masked = plot_merged_data['Replicate_masked'].unique()
n_reps = len(unique_replicates_masked)
ncols = 4
nrows = int(np.ceil(n_reps / ncols))

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols * 6, nrows * 5), squeeze=False)

for idx, replicate_masked in enumerate(unique_replicates_masked):
    ax = axes[idx // ncols][idx % ncols]
    df_rep = plot_merged_data[plot_merged_data['Replicate_masked'] == replicate_masked]
    x = np.log10(df_rep['openswath_lh_ratio'])
    y = np.log10(df_rep['RatioLightToHeavy'])

    # Color by protein name
    colors = df_rep['Protein Name'].map(protein2color)

    scatter = ax.scatter(x, y, c=colors, alpha=0.6, label=None)

    # Add diagonal line with slope 1 (y = x)
    if len(x) > 0 and len(y) > 0:
        xy_min = min(x.min(), y.min())
        xy_max = max(x.max(), y.max())
        ax.plot([xy_min, xy_max], [xy_min, xy_max], color='red', linestyle='--', linewidth=1, label='y = x')

    ax.set_title(f'Replicate: {replicate_masked}')

    # ADD SUBTITLE to subplot (ApoEdge (18 Proteins))
    ax.text(0.5, 1.05, 'ApoEdge (18 Proteins)',
            transform=ax.transAxes, ha='center', va='bottom', fontsize=11, fontweight='normal')

    # Save only the first subplot as the LinkedIn-style plot
    if idx == 0:
        # Save the first subplot in plot/linkedin/ at 1200x627 px (4x2.09 inch at 300 dpi)
        import os
        linkedin_dir = "plot/linkedin"
        os.makedirs(linkedin_dir, exist_ok=True)
        # Extract only the first axes (the current one), ensure tight bounding box
        fig_single, ax_single = plt.subplots(figsize=(4, 2.09), dpi=300)
        # Copy data from ax to ax_single (replot)
        ax_single.scatter(x, y, c=colors, alpha=0.6)
        if len(x) > 0 and len(y) > 0:
            xy_min = min(x.min(), y.min())
            xy_max = max(x.max(), y.max())
            ax_single.plot([xy_min, xy_max], [xy_min, xy_max], color='red', linestyle='--', linewidth=1, label='y = x')
        # Add title and subtitle
        ax_single.set_title(f'Replicate: {replicate_masked}', fontsize=14, pad=16)
        ax_single.text(0.5, 1.01, 'ApoEdge (18 Proteins)', transform=ax_single.transAxes, ha='center', va='bottom', fontsize=12)
        ax_single.set_xlabel('OpenSWATH [log10(Ratio)]')
        ax_single.set_ylabel('Skyline [log10(Ratio)]')
        ax_single.grid(True)
        # Build legend for proteins (same logic as above)
        handles = []
        labels = []
        for prot in unique_proteins[:10]:
            handles.append(plt.Line2D([], [], marker='o', color='w', markerfacecolor=protein2color[prot], markersize=8))
            labels.append(prot)
        if len(unique_proteins) > 10:
            handles.append(plt.Line2D([], [], marker='o', color='w', markerfacecolor="grey", markersize=8))
            labels.append('Other Proteins')
        ax_single.legend(handles, labels, title='Protein Name', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        fig_single.savefig(os.path.join(linkedin_dir, "openswath_vs_skyline_linkedin.png"), dpi=300)
        plt.close(fig_single)
    ax.set_xlabel('OpenSWATH [log10(Ratio)]')
    ax.set_ylabel('Skyline [log10(Ratio)]')
    ax.grid(True)

    # Create custom legend for protein names (show up to 10 for clarity)
    if idx == 0:  # Only add legend to the first plot for clarity
        handles = []
        labels = []
        for prot in unique_proteins[:10]:
            handles.append(plt.Line2D([], [], marker='o', color='w', markerfacecolor=protein2color[prot], markersize=8))
            labels.append(prot)
        if len(unique_proteins) > 10:
            handles.append(plt.Line2D([], [], marker='o', color='w', markerfacecolor="grey", markersize=8))
            labels.append('Other Proteins')
        ax.legend(handles, labels, title='Protein Name', bbox_to_anchor=(1.05, 1), loc='upper left')

# Hide any unused subplots
for idx in range(n_reps, nrows * ncols):
    fig.delaxes(axes[idx // ncols][idx % ncols])

plt.tight_layout()
plt.show()


# Questions for Justin

1. What is the overlap of the identified peptides (heavy and light) between skyline and openswath?
2. For the overlapping peptides between methods, how do the features correlate for retention time and intensity?
3. For the overlapping peptides, are we identifying/using the same set of fragment ions? (you can parse this from the openswath.results.tsv aggr_Fragment_Annotation column). And to your question, yes aggr_Peak_Area is the individual fragment ions intensity, that contributes to the precursors peak-group Intensity.
4. Are we missing some identifications that Skyline identifies? Or do we identify the same things as Skyline, but we chose different features due to the downstream scoring and FDR estimation? You can generate an unfiltered results.tsv if you run: pyprophet export tsv --in openswath_results.osw --out openswath_full.results.tsv --max_rs_peakgroup_qvalue 1 --max_global_peptide_qvalue 1 --max_global_protein_qvalue 1. This will return all the features, so maybe the same Skyline candidate features are identified, we just filtered them out during scoring.
